公共引擎
仅定义函数，不自动下载数据或运行回测。策略、成交规则、指标和选参算法沿用原代码。


In [ ]:
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width',200)

CSV = os.environ.get('ETF_CSV','etf_daily.csv')
TICKS = ['SPY','QQQ','TQQQ','UVXY','TECL','SPXL','SQQQ','TECS','BSV']
ONE_WAY_BPS = {"SPY":0.5, "QQQ":1.5, "TQQQ":1.0, "ES3":10.0, "TECL":35.0,
               "SPXL":16.0, "SQQQ":3.0, "TECS":15.0, "BSV":1.0, "UVXY":6.0, "A35":10.0}
IS_START, IS_END = '2012-08-01', '2019-12-31'
OOS_START = '2020-01-01'

from pathlib import Path
from IPython.display import display

def resolve_csv(csv_path=None):
    explicit = csv_path or os.environ.get('ETF_CSV')
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if path.is_file(): return path
        raise FileNotFoundError(f'找不到数据文件：{path}')
    for root in [Path.cwd(), *Path.cwd().parents]:
        for path in (root / 'data' / 'raw' / 'etf_daily.csv', root / 'data' / 'etf_daily.csv', root / 'etf_daily.csv'):
            if path.is_file(): return path.resolve()
    raise FileNotFoundError('找不到 data/raw/etf_daily.csv，请设置 CSV_PATH 为数据的绝对路径。')

def load(csv_path=None):
    df = pd.read_csv(resolve_csv(csv_path), header=[0,1], index_col=0, skiprows=[2])
    df.index = pd.to_datetime(df.index)
    df = df[df['Close']['SPY'].notna()]
    if df.empty or not df.index.is_monotonic_increasing or df.index.has_duplicates:
        raise ValueError('数据不能为空，且日期必须升序、无重复。')
    required = [(field, ticker) for field in ['Adj Close','Close','Open','High','Low'] for ticker in TICKS]
    missing = [key for key in required if key not in df.columns]
    if missing: raise ValueError(f'CSV 缺少必需列：{missing}')
    return df


In [2]:
def wilder_rsi(x, n=10):
    x = np.asarray(x, float); out = np.full(len(x), np.nan)
    d = np.diff(x)
    # first valid index
    valid = np.where(~np.isnan(x))[0]
    if len(valid) < n+1: return out
    s = valid[0]
    g = np.where(d > 0, d, 0.0); l = np.where(d < 0, -d, 0.0)
    ag = g[s:s+n].mean(); al = l[s:s+n].mean()
    out[s+n] = 100 - 100/(1+ag/al) if al > 0 else 100
    for i in range(s+n+1, len(x)):
        ag = (ag*(n-1) + g[i-1])/n; al = (al*(n-1) + l[i-1])/n
        out[i] = 100 - 100/(1+ag/al) if al > 0 else 100
    return out

def prep(csv_path=None):
    df = load(csv_path)
    adj = df["Adj Close"][TICKS]; cl = df["Close"][TICKS]
    f = adj/cl
    O = df["Open"][TICKS]*f; H = df["High"][TICKS]*f; L = df["Low"][TICKS]*f; C = adj
    return O, H, L, C


In [3]:
def signals(C, p=None):
    p = dict(rsi_n=10, spy_sma=120, qqq_sma=20, tqqq_sma=20, qqq_rsi_bull=81, spy_rsi_bull=80,
             tqqq_rsi_dip=30, uvxy_hi=74, uvxy_ext=84, sqqq_exit=34) | (p or {})
    rsi = {t: pd.Series(wilder_rsi(C[t].values, p["rsi_n"]), index=C.index) for t in TICKS}
    sma = lambda t, n: C[t].rolling(n).mean()
    s_spy, s_qqq, s_tqqq = sma("SPY",p["spy_sma"]), sma("QQQ",p["qqq_sma"]), sma("TQQQ",p["tqqq_sma"])
    tg = []
    for i, d in enumerate(C.index):
        vals = [rsi[t].iloc[i] for t in TICKS] + [s_spy.iloc[i], s_qqq.iloc[i], s_tqqq.iloc[i]]
        if np.isnan(vals).any(): tg.append(None); continue
        r = {t: rsi[t].iloc[i] for t in TICKS}
        pspy, pqqq, ptq = C["SPY"].iloc[i], C["QQQ"].iloc[i], C["TQQQ"].iloc[i]
        mx = lambda a: max(a, key=lambda t: r[t])
        if pspy > s_spy.iloc[i]:
            t = "UVXY" if (r["QQQ"] > p["qqq_rsi_bull"] or r["SPY"] > p["spy_rsi_bull"]) else "TQQQ"
        else:
            if r["TQQQ"] < p["tqqq_rsi_dip"]: t = "TECL"
            elif r["SPY"] < 30: t = "SPXL"
            elif r["UVXY"] > p["uvxy_hi"]:
                if r["UVXY"] > p["uvxy_ext"]:
                    if pqqq > s_qqq.iloc[i]: t = "TECS" if r["SQQQ"] < 31 else "TECL"
                    else: t = mx(["TECS","BSV"])
                else: t = "UVXY"
            else:
                if ptq > s_tqqq.iloc[i]: t = "TECS" if r["SQQQ"] < p["sqqq_exit"] else "TECL"
                else: t = mx(["TECS","BSV"])
        tg.append(t)
    return pd.Series(tg, index=C.index)


In [ ]:
def backtest(O,H,L,C,tg, sl=None, tp=None, trailing=True, start=IS_START):
    idx = C.index; n = len(idx)
    Oa,Ha,La,Ca = O.values,H.values,L.values,C.values
    col = {t:i for i,t in enumerate(TICKS)}
    cash, qty, pos = 1.0, 0.0, None
    entry = peak = None; blocked = None
    eq = np.full(n, np.nan); trades = []; ent_eq = None; pending = None  # target decided at prev close
    started = False
    for k in range(n):
        d = idx[k]
        if not started:
            if d >= pd.Timestamp(start) and tg.iloc[k] is not None: started = True
            else: continue
        # 1) execute pending order at open
        if pending is not None and pending != pos:
            j = col[pending]
            if pos is not None:
                pj = col[pos]; c = ONE_WAY_BPS[pos]/1e4
                cash = qty*Oa[k,pj]*(1-c); qty = 0
                trades.append(cash/ent_eq - 1); pos = None
            c = ONE_WAY_BPS[pending]/1e4
            ent_eq = cash; qty = cash*(1-c)/Oa[k,j]; cash = 0; pos = pending
            entry = peak = Oa[k,j]
        pending = None
        # 2) intraday stop / take-profit
        if pos is not None:
            j = col[pos]; o,h,l = Oa[k,j],Ha[k,j],La[k,j]
            ref = peak if trailing else entry
            stop = ref*(1-sl) if sl else -np.inf
            tpl = entry*(1+tp) if tp else np.inf
            fill = None
            if o >= tpl: fill = o
            elif o <= stop: fill = o
            elif l <= stop: fill = stop
            elif h >= tpl: fill = tpl
            if fill is not None:
                c = ONE_WAY_BPS[pos]/1e4
                cash = qty*fill*(1-c); qty = 0
                trades.append(cash/ent_eq - 1)
                blocked = pos; pos = None
            else:
                peak = max(peak, h)
        eq[k] = cash + (qty*Ca[k,col[pos]] if pos else 0)
        # 3) close: signal -> order for next open
        t = tg.iloc[k]
        if t is not None:
            if t == blocked: pending = None
            else:
                blocked = None
                pending = t if t != pos else None
    s = pd.Series(eq, index=idx).dropna()
    return s, trades


In [5]:
def metrics(eq, trades=None, a=None, b=None):
    e = eq.loc[a:b] if a or b else eq
    r = e.pct_change().dropna()
    yrs = (e.index[-1]-e.index[0]).days/365.25
    cagr = (e.iloc[-1]/e.iloc[0])**(1/yrs)-1
    vol = r.std()*np.sqrt(252); sh = r.mean()*252/vol if vol>0 else np.nan
    dn = np.sqrt((np.minimum(r,0)**2).mean())*np.sqrt(252)
    mdd = (e/e.cummax()-1).min()
    out = dict(CAGR=cagr, Vol=vol, Sharpe=sh, Sortino=r.mean()*252/dn if dn>0 else np.nan, MDD=mdd, Calmar=cagr/abs(mdd) if mdd<0 else np.nan)
    return out

def trade_stats(trades):
    if not trades: return dict(N=0, Win=np.nan, PL=np.nan)
    t = np.array(trades); w = t[t>0]; l = t[t<=0]
    return dict(N=len(t), Win=len(w)/len(t), PL=(w.mean()/abs(l.mean()) if len(w) and len(l) else np.nan))


In [ ]:
# ---- 网格 / 选参函数 ----
SLs=[0.06,0.08,0.10,0.12,None]; TPs=[0.15,0.25,0.40,None]
combos=[(tr,sl,tp) for tr in [True,False] for sl in SLs for tp in TPs if not (sl is None and not tr)]
base=(True,None,None)   # 基线：无止损无止盈

def run_all(O, H, L, C, tg):
    return {c: backtest(O,H,L,C,tg,c[1],c[2],c[0]) for c in combos}

def neigh(c, score):
    """邻域平均：参数网格上相邻 3x3 格子的平均得分，避免选到孤立最优点"""
    tr,sl,tp=c; i=SLs.index(sl); j=TPs.index(tp); vals=[]
    for di in (-1,0,1):
        for dj in (-1,0,1):
            ii,jj=i+di,j+dj
            if 0<=ii<len(SLs) and 0<=jj<len(TPs):
                k=(tr,SLs[ii],TPs[jj])
                if k in score: vals.append(score[k])
                elif SLs[ii] is None: vals.append(score.get((True,None,TPs[jj]),np.nan))
    return np.nanmean(vals)

def select(runs, a, b):
    """只用 [a,b] 区间选参：回撤不劣于基线，按邻域平均 Sharpe 选最优"""
    sc={c:metrics(runs[c][0],a=a,b=b) for c in combos}
    shp={c:sc[c]['Sharpe'] for c in combos}
    ok=[c for c in combos if sc[c]['MDD']>=sc[base]['MDD'] and c!=base]
    return max(ok,key=lambda c:neigh(c,shp)), sc

fmt=lambda m:{k:round(float(v),3) for k,v in m.items()}
def label(c): return ('追踪' if c[0] else '固定')+f' 止损={c[1]} 止盈={c[2]}'


def prepare_context(csv_path=None):
    """一次读取数据、复权并生成信号，显式返回给各步骤。"""
    O, H, L, C = prep(csv_path)
    tg = signals(C)
    print('数据区间:', C.index[0].date(), '~', C.index[-1].date(), '| 信号分布:', tg.value_counts().to_dict())
    return O, H, L, C, tg
